## Scraper de TimingSys / Cronomancha (timingsys.com)

`timingsys.com` es en realidad la web pública de **Cronomancha**, la empresa de
cronometraje (el propio `<title>` de la web es "Cronomancha"). Publica un
catálogo completo y navegable de todas las carreras que ha cronometrado, en
`https://timingsys.com/results?page=N` — listado paginado (10 eventos por
página; a fecha de escribir este notebook hay **59 páginas**, ~590 eventos),
cada uno con nombre, fecha, lugar, inscritos y, si ya tiene resultados
publicados, un botón "Resultados" que enlaza a `resultdata/{id}`.

Igual que en el notebook de un solo evento que hicimos antes, esa página
`resultdata/{id}` es solo una carcasa HTML: el botón "Resultados/Directo" enlaza
a una SPA externa (`resultados-app.cronomancha.com`) que a su vez llama a una
**API JSON pública sin autenticación**:

```
GET https://resultados-api.cronomancha.com/api/v1/public/events/{hash_evento}
GET https://resultados-api.cronomancha.com/api/v1/public/races/{slug_carrera}/results?limit=0
```

**Aviso importante que no vimos con el evento 757:** `{hash_evento}` no
siempre es un hash hexadecimal corto — en eventos con una sola carrera/prueba
(la mayoría del catálogo) es directamente el **slug** de la carrera (p. ej.
`xxi-cp-santa-eulalia-villares-del-saz-2026`), y esa misma API devuelve
`"type": "race"` en vez de `"type": "event"`. La regex de resolución tiene que
aceptar cualquier cadena, no solo hexadecimal.

Mismo esquema en dos niveles que en Kronoak/RaceResult, con checkpoint como en
cronorunner/mychip/raceresult (esto es un catálogo grande, con permitir
interrumpir y continuar es imprescindible):

1. **Nivel 1 — catálogo**: recorremos `results?page=N` para todas las páginas
   (el nº total se detecta solo, leyendo la paginación) y extraemos de cada
   tarjeta: id, nombre, fecha, lugar, inscritos, precio y si tiene botón de
   resultados. Se guarda en `timingsys_catalogo_eventos.csv` con checkpoint por
   página.
2. **Nivel 2 — resultados**: para cada evento del catálogo con resultados
   publicados (y que no esté ya en el checkpoint), resolvemos el
   `hash_evento`/slug en su página `resultdata/{id}` y pedimos a la API de
   Cronomancha todas sus carreras y el listado completo de participantes de
   cada una. Se va añadiendo a `DF_TIMINGSYS_SUCIO.csv` evento a evento, con
   checkpoint en `timingsys_eventos_checkpoint.json` para poder interrumpir y
   continuar sin repetir peticiones.

**Nota sobre la ejecución de este notebook:** el entorno donde lo hemos escrito
no tiene acceso de red real a `timingsys.com` ni a `cronomancha.com` (mismo
caso que con `cronorunner`), así que hemos validado la lógica de las dos fases
a través de un navegador real: la página 1 completa del catálogo (10 eventos,
ver más abajo) y dos eventos con resultados reales (757 y 797, con sus 74 y 89
filas respectivamente, ya escritos en `DF_TIMINGSYS_SUCIO.csv`). **No se ha
lanzado aquí a escala completa** — son ~590 eventos y, de los que tengan
resultados publicados, cientos de peticiones a la API. En tu máquina, con
salida a internet normal, `descargar_catalogo_completo()` y
`descargar_resultados_pendientes()` recorren el catálogo completo; ambas
tienen checkpoint propio, así que se pueden interrumpir (Kernel → Interrupt) y
volver a lanzar sin perder lo ya hecho.

In [6]:
import re
import json
import io
import zipfile
import unicodedata
from datetime import datetime
import time
import urllib.parse
from pathlib import Path

import pandas as pd
import requests

# ==============================================================================
# 1. CONFIGURACIÓN DEL ENTORNO
# ==============================================================================
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "es-ES,es;q=0.9,ca;q=0.8",
}

URL_LISTADO = "https://timingsys.com/results"
API_CRONOMANCHA = "https://resultados-api.cronomancha.com/api/v1/public"

OUTPUT_DIR = Path("../../data/raw/timingsys")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_CATALOGO = OUTPUT_DIR / "timingsys_catalogo_eventos.csv"
CSV_RESULTADOS = OUTPUT_DIR / "DF_TIMINGSYS_SUCIO.csv"
CHECKPOINT_EVENTOS = OUTPUT_DIR / "timingsys_eventos_checkpoint.json"

# Regex sobre el botón "Resultados/Directo" de la página resultdata/{id}, del
# tipo <a ... href="https://resultados-app.cronomancha.com/results/HASH?race=SLUG">.
# HASH puede ser un hash hexadecimal corto (eventos con varias carreras, p.ej.
# el 757) o directamente el slug de la carrera (eventos con una sola prueba,
# la mayoría) -- por eso el primer grupo acepta cualquier carácter que no sea
# "/", "?" o comilla, en vez de limitarse a [a-f0-9]+.
# Antes exigíamos "...?race=SLUG" al final, pero muchos eventos enlazan
# directamente a ".../results/HASH" sin ese parámetro (el ?race= solo
# preselecciona una carrera en la SPA; a nosotros nos basta con el hash
# para pedir /events/{hash} y traer TODAS las carreras del evento).
RE_ENLACE_CRONOMANCHA = re.compile(
    r'resultados-app\.cronomancha\.com/results/([^/?"\']+)',
    re.IGNORECASE,
)

_MESES_ABREV = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12,
}
_RE_FECHA_CATALOGO = re.compile(r"(\d{1,2})\s+(\w{3})\s+(\d{4})")

# Cada evento tiene una página de ubicación (/event/{id}/location) con un mapa
# Leaflet. Si el organizador ha marcado un punto, el HTML incluye una llamada
# `setView([lat, lon], zoom)`; si no, el mapa ni se inicializa (no hay match).
RE_COORDENADAS = re.compile(r"setView\(\[(-?\d+\.\d+),\s*(-?\d+\.\d+)\]")

# Sistema LEGACY de Cronomancha (dominio sin "-app", resultados.cronomancha.com):
# usado por bastantes eventos, sobre todo anteriores a 2026. El botón
# "Resultados/Directo" enlaza a una SPA con dos formatos de URL vistos
# ("#/carrera/SLUG" y el más antiguo "#!/SLUG/"), pero ambos apuntan al mismo
# backend estático: METADATOS en `data/SLUG/index.json` y participantes en un
# `data/SLUG/{TS}_result.json.zip` (ZIP estándar con un único JSON dentro,
# comprimido en deflate -- zipfile de la librería estándar lo lee sin problema,
# validado a mano contra un evento real: 205 participantes, 185 finishers,
# 17 DNS y 3 DNF, coincide exactamente con lo que muestra la web).
LEGACY_CRONOMANCHA = "https://resultados.cronomancha.com"
RE_LEGACY_SLUG = re.compile(
    r'href="https?://resultados\.cronomancha\.com/#(?:/carrera/|!/)([^"/?]+)/?"',
    re.IGNORECASE,
)

# Pausas de cortesía entre peticiones.
PAUSA_LISTADO = 0.5
PAUSA_TIMINGSYS = 0.6
PAUSA_API = 0.4


def obtener_html(url, timeout=20):
    """Descarga una URL y devuelve el HTML, o None si falla."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print(f"  \U0001f6a8 Error al descargar {url}: {e}")
        return None


def obtener_json(url, timeout=20):
    """Descarga una URL y la interpreta como JSON, o devuelve None si falla."""
    try:
        resp = requests.get(url, headers={**HEADERS, "Accept": "application/json"}, timeout=timeout)
        resp.raise_for_status()
        return resp.json()
    except (requests.RequestException, ValueError) as e:
        print(f"  \U0001f6a8 Error al descargar/parsear JSON de {url}: {e}")
        return None


def fecha_catalogo_a_iso(texto_fecha):
    """Convierte '9 ago 2026' en '2026-08-09'. Devuelve None si no matchea."""
    if not texto_fecha:
        return None
    m = _RE_FECHA_CATALOGO.search(texto_fecha.lower())
    if not m:
        return None
    dia, mes_abrev, anio = m.groups()
    mes = _MESES_ABREV.get(mes_abrev)
    return f"{anio}-{mes:02d}-{int(dia):02d}" if mes else None

In [7]:
# ==============================================================================
# INTERRUPTOR PRUEBA RÁPIDA / DESCARGA COMPLETA
# ==============================================================================
# True  -> prueba rápida: solo 2 páginas de catálogo (~20 eventos) y máximo
#          5 eventos nuevos con resultados. Para comprobar esta noche que todo
#          sigue funcionando antes de lanzar la descarga completa.
# False -> descarga completa: las 59 (o las que haya) páginas de catálogo y
#          TODOS los eventos con resultados pendientes.
#
# Para la prueba de esta noche: déjalo en True, ejecuta todo el notebook
# (Run All) y revisa que las dos celdas de "Ejecución" imprimen resultados
# razonables. Si todo va bien, cambia a False y vuelve a ejecutar todo el
# notebook para la descarga completa (puede tardar bastante: son ~590
# eventos de catálogo y varios cientos de eventos con resultados).
MODO_PRUEBA = False

MAX_PAGINAS_CATALOGO = 2 if MODO_PRUEBA else 200
MAX_EVENTOS_RESULTADOS = 5 if MODO_PRUEBA else None

print(f"MODO_PRUEBA={MODO_PRUEBA} -> "
      f"máx. {MAX_PAGINAS_CATALOGO} página(s) de catálogo, "
      f"máx. {MAX_EVENTOS_RESULTADOS or 'todos los'} evento(s) con resultados")

MODO_PRUEBA=False -> máx. 200 página(s) de catálogo, máx. todos los evento(s) con resultados


### Nivel 1 — catálogo paginado de eventos

Cada tarjeta de evento es un bloque `<div class="box_left col-sm" ...>` que
contiene, entre otras cosas, dos `<h3>`: uno oculto ("VER EVENTO", el overlay
que aparece al pasar el ratón sobre la imagen) y el real, dentro de
`right_top`. Los campos numéricos/de texto ("Inscritos", "Distancia",
"Precio desde") tienen una particularidad del HTML de esta web: el valor no
va *dentro* del `<span class="event_details">`, sino justo *después* de su
etiqueta de cierre — así que la función de extracción prueba las dos
posiciones. Verificado a mano contra la página 1 real (ver la celda de
ejecución más abajo).

In [8]:
def extraer_campo(card, etiqueta):
    """Busca 'etiqueta</span><span class="event_details">VALOR</span>RESTO' y
    devuelve VALOR si no está vacío, o RESTO en caso contrario -- en esta web
    el valor real puede estar dentro del span (p.ej. 'Lugar') o justo después
    de un span vacío (p.ej. 'Inscritos', 'Distancia', 'Precio desde')."""
    patron = re.escape(etiqueta) + r'\s*</span>\s*<span class="event_details[^"]*">([^<]*)</span>\s*([^<]*)'
    m = re.search(patron, card)
    if not m:
        return None
    v1, v2 = m.group(1).strip(), m.group(2).strip()
    return v1 or v2 or None


def parsear_pagina_catalogo(html):
    """Divide el HTML de una página de resultados en tarjetas de evento y
    extrae los campos de cada una."""
    cards = html.split('<div class="box_left col-sm"')[1:]
    eventos = []
    for card in cards:
        m_res = re.search(r"resultdata/(\d+)", card)
        m_ev = re.search(r"event/(\d+)", card)
        m_nombre = re.search(r'right_top">\s*<a[^>]*>\s*<h3>([^<]+)</h3>', card)
        id_evento = m_res.group(1) if m_res else (m_ev.group(1) if m_ev else None)
        if id_evento is None:
            continue
        fecha_texto = extraer_campo(card, "Fecha del Evento")
        eventos.append({
            "id_timingsys": int(id_evento),
            "url_timingsys": f"https://timingsys.com/resultdata/{id_evento}",
            "nombre_evento": (re.sub(r"\s+", " ", m_nombre.group(1)).strip() if m_nombre else None),
            "fecha_texto": fecha_texto,
            "fecha_evento": fecha_catalogo_a_iso(fecha_texto),
            "distancia_texto": extraer_campo(card, "Distancia: "),
            "lugar": extraer_campo(card, "Lugar: "),
            "inscritos": extraer_campo(card, "Inscritos: "),
            "precio_desde": extraer_campo(card, "Precio desde: "),
            "tiene_resultados": bool(m_res),
        })
    return eventos


def detectar_num_paginas(html):
    """Lee el número de página más alto enlazado en la paginación (incluye el
    último enlace numerado, que es el total real de páginas)."""
    numeros = [int(n) for n in re.findall(r"results\?page=(\d+)", html)]
    return max(numeros) if numeros else 1

In [9]:
def descargar_catalogo_completo(max_paginas_seguridad=200):
    """Recorre todo el catálogo paginado con checkpoint: si ya existe
    `timingsys_catalogo_eventos.csv`, retoma desde la última página guardada
    en vez de descargar todo de cero."""
    eventos_previos = []
    pagina_inicio = 1
    if CSV_CATALOGO.exists():
        df_prev = pd.read_csv(CSV_CATALOGO)
        eventos_previos = df_prev.to_dict("records")
        pagina_inicio = int(df_prev["pagina"].max()) + 1
        print(f"Checkpoint: {len(eventos_previos)} eventos ya en catálogo, "
              f"retomando en página {pagina_inicio}")

    html_pagina_1 = obtener_html(f"{URL_LISTADO}?page=1")
    num_paginas = detectar_num_paginas(html_pagina_1) if html_pagina_1 else 1
    print(f"Total de páginas detectadas: {num_paginas} (~{num_paginas * 10} eventos aprox.)")

    eventos = list(eventos_previos)
    for pagina in range(pagina_inicio, min(num_paginas, max_paginas_seguridad) + 1):
        html = html_pagina_1 if pagina == 1 else obtener_html(f"{URL_LISTADO}?page={pagina}")
        if html is None:
            continue
        nuevos = parsear_pagina_catalogo(html)
        for ev in nuevos:
            ev["pagina"] = pagina
        eventos.extend(nuevos)
        print(f"  \u21b3 Página {pagina}/{num_paginas}: {len(nuevos)} eventos")

        df_eventos = pd.DataFrame(eventos).drop_duplicates(subset="id_timingsys", keep="last")
        df_eventos.to_csv(CSV_CATALOGO, index=False, encoding="utf-8-sig")
        time.sleep(PAUSA_LISTADO)

    return pd.DataFrame(eventos).drop_duplicates(subset="id_timingsys", keep="last")

### Ejecución Nivel 1

Controlada por el interruptor `MODO_PRUEBA` de arriba: con `True` descarga solo `MAX_PAGINAS_CATALOGO` páginas (prueba rápida de esta noche); con `False`, el catálogo completo. Es la misma llamada en los dos casos, así que no hay que tocar nada más para pasar de una a otra -- solo el interruptor.

In [10]:
df_catalogo = descargar_catalogo_completo(max_paginas_seguridad=MAX_PAGINAS_CATALOGO)
print(df_catalogo[["id_timingsys", "nombre_evento", "fecha_evento", "inscritos", "tiene_resultados"]])

Checkpoint: 530 eventos ya en catálogo, retomando en página 54
Total de páginas detectadas: 59 (~590 eventos aprox.)
  ↳ Página 54/59: 10 eventos
  ↳ Página 55/59: 10 eventos
  ↳ Página 56/59: 10 eventos
  ↳ Página 57/59: 10 eventos
  ↳ Página 58/59: 10 eventos
  ↳ Página 59/59: 10 eventos
     id_timingsys                                      nombre_evento  \
0             729  PR-CU 23 "Ruta del Covacho y El Callejón" - So...   
1             767                        XXI CP Santa Eulalia - 2026   
2             766                        V CP La Joya Barroca - 2026   
3             765                         XXII Trofeo San Gil - 2026   
4             797           IX LEGUA ALMARCHEÑA - LA ALMARCHA - 2026   
..            ...                                                ...   
585           365                             BASE EVENT - CCPP 2023   
586           477                             BASE EVENT - CCPP 2024   
587           593                             BASE EVENT - CC

### Nivel 2 — resolución de hash/slug, ubicación y resultados agregados de cada evento

Para cada evento del catálogo con `tiene_resultados=True` que no esté ya en el
checkpoint: descargamos su `resultdata/{id}`, sacamos el `hash_evento` (o slug)
del botón "Resultados/Directo", pedimos a la API todas las carreras del evento
y, para cada una, el listado completo de participantes con `limit=0`.

**El botón de TimingSys no siempre lleva un hash/slug en la URL.** Comprobado
con la ejecución real de esta noche: muchos eventos (sobre todo circuitos y
clubes) enlazan solo a la portada `resultados-app.cronomancha.com/` sin nada
más, y antes los marcábamos como "sin enlace a Cronomancha" y los saltábamos
-- pero sí que están en Cronomancha, solo que la URL no lo dice. Ahora, cuando
pasa esto, buscamos el nombre del evento (el mismo `<h1>` de la página) en
`GET {API_CRONOMANCHA}/events`, el listado de eventos recientes que alimenta
esa portada, comparando nombres normalizados (mayúsculas, sin acentos, sin
puntuación). Es un fallback *best-effort*: ese listado solo cubre los eventos
más recientes (no tiene paginación), así que un evento antiguo puede seguir
sin encontrarse y se sigue saltando igual que antes -- pero recupera una parte
real de los que antes se perdían sin necesidad de tocar nada más.

También ampliamos el regex principal: antes exigíamos `...results/HASH?race=
SLUG`, pero hay eventos que enlazan directamente a `.../results/HASH` sin el
`?race=`, y esos también se estaban saltando por error de regex, no porque
falten datos.

**Actualización tras la descarga completa de esta noche:** la mayoría de los
eventos que salían como "sin enlace a Cronomancha" en realidad sí tenían
resultados, pero en un segundo sistema de Cronomancha, el **legacy**
(`resultados.cronomancha.com`, sin "-app" en el dominio) -- un backend
estático distinto de la API REST del sistema nuevo, con los datos de cada
carrera en `data/{slug}/index.json` (metadatos) y `data/{slug}/{TS}_result.json.zip`
(participantes, un ZIP estándar con un único JSON dentro). `resolver_enlace_cronomancha`
detecta ahora los dos formatos de enlace del legacy (`#/carrera/SLUG` y el más
antiguo `#!/SLUG/`) y `filas_de_evento_legacy` agrega sus resultados exactamente
igual que el sistema nuevo (`finisher_h`/`finisher_d`/`total_finishers`, nunca
datos de participante individual). A diferencia del sistema nuevo (donde un
evento puede tener varias carreras), en el legacy cada slug es siempre una
única carrera/distancia.

Quedan **fuera de este fix**, porque son sistemas de cronometraje distintos y
requerirían un scraper aparte: los dominios de circuitos propios
(`mtb.cronomancha.es`, `circuitos2026.cronomancha.es`, `ccpp.cronomancha.es`) y
las páginas de resultados en vivo propias de TimingSys
(`timingsys.com/eventdata/pdf/.../*.html`). Se quedan marcados como "sin
enlace a Cronomancha" igual que antes.

Además, TimingSys tiene una página de ubicación por evento
(`/event/{id}/location`) con un mapa Leaflet: cuando el organizador ha marcado
un punto, el HTML incluye `setView([lat, lon], zoom)` con las coordenadas
reales del punto de salida/meta -- mucha más información útil que el campo de
texto libre "Lugar" del catálogo (que muchos organizadores dejan en blanco).
Sacamos `lat`/`lon` una vez por evento y las repetimos en cada fila de carrera,
igual que hace `scrape_raceresult` (`raceresult_ubicaciones.csv`): en la
limpieza posterior se pueden geocodificar a `municipio`/`comarca`/`provincia`
del mismo modo.

**No se guarda ninguna fila a nivel de participante individual.** Igual que en
`Scraper_Kronoak` y en `scrape_cronorunner` (una fila por carrera con
`finisher_h`/`finisher_d`, nunca dorsales ni nombres), aquí agregamos en el
momento: por cada carrera del evento contamos cuántos participantes con
`estado == "FINISHED"` son `sexo == "M"` (`finisher_h`) y cuántos `sexo == "F"`
(`finisher_d`), y esas son las únicas columnas que salen de Nivel 2 aparte de
los metadatos de la carrera y las coordenadas. Se descarta todo lo demás
(nombre, dorsal, categoría, equipo, tiempo, posición...): no hace falta para
el TFM y no tiene sentido guardarlo.

**Segunda mejora tras revisar los datos:** cuando un evento de TimingSys combina varias distancias en una sola tarjeta (p.ej. "Media maratón y 10K"), el sistema legacy las guarda como carreras independientes pero TimingSys solo enlaza a una -- se estaba perdiendo la otra por completo. `buscar_carreras_legacy_por_nombre_fecha` usa el índice global del sistema legacy (`data/index.php`, con el histórico completo de carreras) para encontrar, por fecha exacta y nombre contenido, las carreras "hermanas" de un mismo evento, y `resolver_enlace_cronomancha` las procesa todas. Comprobado con un caso real: el evento combinado tenía 990 inscritos pero la carrera enlazada directamente solo cubría 352 (el 21K); el 10K -- con el resto de inscritos -- solo aparecía en el índice global.


In [11]:
_CACHE_EVENTOS_GLOBALES = None
_CACHE_INDICE_LEGACY_GLOBAL = None


def _indice_legacy_global():
    """Cachea (una vez por ejecución) el índice COMPLETO de carreras del
    sistema legacy de Cronomancha (`data/index.php`, ~1200 carreras, todo el
    histórico -- a diferencia del listado de eventos recientes del sistema
    nuevo, este sí cubre carreras antiguas). Se usa para encontrar las
    carreras "hermanas" de un mismo evento: cuando TimingSys combina varias
    distancias en una sola tarjeta (p.ej. "Media maratón y 10K") pero solo
    enlaza a una, Cronomancha las guarda como carreras legacy independientes
    (slugs distintos) que este índice permite encontrar por fecha y nombre."""
    global _CACHE_INDICE_LEGACY_GLOBAL
    if _CACHE_INDICE_LEGACY_GLOBAL is None:
        data = obtener_json(f"{LEGACY_CRONOMANCHA}/data/index.php")
        _CACHE_INDICE_LEGACY_GLOBAL = data or []
    return _CACHE_INDICE_LEGACY_GLOBAL


def buscar_carreras_legacy_por_nombre_fecha(nombre_evento_ts, fecha_evento_ts_iso):
    """Busca en el índice global del sistema legacy todas las carreras que
    caigan en la misma fecha que el evento de TimingSys y cuyo nombre esté
    "contenido" en el del evento (todas las palabras del nombre legacy
    aparecen entre las del nombre de TimingSys, comparando en mayúsculas y
    sin acentos/puntuación). Devuelve la lista de slugs (`dir`) que
    cumplen las dos condiciones, sin duplicados.

    Validado a mano contra un caso real: el evento de TimingSys "26 MEDIA
    MARATON DE CUENCA Y 10K - 2025" solo enlaza a la carrera legacy de la
    media maratón (352 corredores), pero el 10K es una carrera legacy
    aparte ("10k-cuenca-2025", con los ~640 corredores restantes de los
    990 inscritos totales) que esta búsqueda sí encuentra."""
    if not fecha_evento_ts_iso:
        return []
    tokens_ts = set(_normaliza(nombre_evento_ts).split())
    if not tokens_ts:
        return []
    slugs = []
    vistos = set()
    for carrera in _indice_legacy_global():
        fecha_iso = _fecha_legacy_a_iso(carrera.get("FechaCarreraT01"))
        if fecha_iso != fecha_evento_ts_iso:
            continue
        tokens_carrera = set(_normaliza(carrera.get("NombreCarreraT01")).split())
        # Al menos 2 palabras para evitar coincidencias triviales (p.ej. solo
        # el año), y que TODAS estén contenidas en el nombre de TimingSys.
        if len(tokens_carrera) < 2 or not tokens_carrera.issubset(tokens_ts):
            continue
        slug = carrera.get("dir")
        if slug and slug not in vistos:
            vistos.add(slug)
            slugs.append(slug)
    return slugs


def _normaliza(texto):
    """Mayúsculas, sin acentos, sin puntuación -- para comparar nombres de
    evento entre TimingSys y Cronomancha de forma tolerante a pequeñas
    diferencias de formato."""
    if not texto:
        return ""
    texto = unicodedata.normalize("NFD", texto.upper())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return re.sub(r"[^A-Z0-9]+", " ", texto).strip()


def _eventos_globales_cronomancha():
    """Cachea (una vez por ejecución) el listado de eventos recientes de
    Cronomancha -- el mismo que alimenta la portada de
    resultados-app.cronomancha.com. Solo cubre los eventos más recientes (no
    hay paginación ni parámetros que lo cambien, comprobado), así que es un
    fallback best-effort, no sustituye la resolución normal por hash/slug."""
    global _CACHE_EVENTOS_GLOBALES
    if _CACHE_EVENTOS_GLOBALES is None:
        data = obtener_json(f"{API_CRONOMANCHA}/events")
        _CACHE_EVENTOS_GLOBALES = data or []
    return _CACHE_EVENTOS_GLOBALES


def buscar_hash_por_nombre(nombre_evento_ts):
    """Cuando el botón de TimingSys enlaza solo a la portada de Cronomancha
    (sin hash/slug en la URL), busca el evento por nombre normalizado en el
    listado global de eventos recientes. Devuelve el slug/hash si hay
    coincidencia exacta o de contención, o None si no aparece (evento más
    antiguo que la ventana del listado, u otra discrepancia de nombre)."""
    objetivo = _normaliza(nombre_evento_ts)
    if not objetivo:
        return None
    for ev in _eventos_globales_cronomancha():
        n = _normaliza(ev.get("name"))
        if n and (n == objetivo or objetivo in n or n in objetivo):
            return ev.get("slug")
    return None


def resolver_enlace_cronomancha(id_evento, nombre_evento_ts=None, fecha_evento_ts_iso=None):
    """Descarga la página de TimingSys de un evento e identifica en qué
    sistema de Cronomancha están sus resultados: el nuevo
    (resultados-app.cronomancha.com, API REST) o el legacy
    (resultados.cronomancha.com, datos en un .zip por carrera -- ver celda de
    arriba). Devuelve (sistema, referencia) con sistema en
    {"nuevo", "legacy", None}.

    Para "nuevo", referencia es un único hash/slug (el propio sistema nuevo
    ya agrupa todas las carreras del evento en su API). Para "legacy",
    referencia es una LISTA de slugs: el enlazado directamente desde
    TimingSys más, si se pasan `nombre_evento_ts`/`fecha_evento_ts_iso`,
    cualquier carrera "hermana" que el índice global del sistema legacy
    tenga en la misma fecha con un nombre contenido en el del evento --
    así se recuperan las distancias que TimingSys combina en una sola
    tarjeta pero Cronomancha guarda como carreras independientes (ver
    `buscar_carreras_legacy_por_nombre_fecha` más arriba)."""
    html = obtener_html(f"https://timingsys.com/resultdata/{id_evento}")
    if html is None:
        return None, None

    m = RE_ENLACE_CRONOMANCHA.search(html)
    if m:
        return "nuevo", m.group(1)

    m = RE_LEGACY_SLUG.search(html)
    if m:
        slug_principal = m.group(1)
        hermanas = (
            buscar_carreras_legacy_por_nombre_fecha(nombre_evento_ts, fecha_evento_ts_iso)
            if nombre_evento_ts else []
        )
        slugs = [slug_principal] + [s for s in hermanas if s != slug_principal]
        return "legacy", slugs

    # El botón puede enlazar solo a la portada del sistema nuevo, sin
    # hash/slug en la URL. Fallback: buscar el nombre del evento en el
    # listado global reciente de resultados-app.cronomancha.com.
    if "resultados-app.cronomancha.com" in html:
        m_nombre = re.search(r"<h1>Resultados - ([^<]+)</h1>", html)
        if m_nombre:
            slug = buscar_hash_por_nombre(m_nombre.group(1))
            if slug:
                return "nuevo", slug

    return None, None


def obtener_coordenadas_evento(id_evento):
    """Descarga la página de ubicación del evento en TimingSys
    (`/event/{id}/location`) y devuelve (lat, lon) si el organizador ha
    marcado un punto en el mapa Leaflet, o (None, None) si no -- en ese caso
    el mapa simplemente no se inicializa en el HTML (comprobado con varios
    eventos reales, no es un fallo de parseo)."""
    html = obtener_html(f"https://timingsys.com/event/{id_evento}/location")
    if html is None:
        return None, None
    m = RE_COORDENADAS.search(html)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)


def obtener_carreras_evento(hash_evento):
    """Metadatos del evento (nombre, fecha...) y lista de todas sus carreras."""
    data = obtener_json(f"{API_CRONOMANCHA}/events/{hash_evento}")
    if not data:
        return None, []
    info_evento = {
        "nombre_evento": data.get("name"),
        "fecha_evento": data.get("date_start"),
    }
    return info_evento, data.get("races", [])


def obtener_resultados_carrera(slug_carrera):
    """Listado completo de resultados de una carrera. limit=0 devuelve todos
    los participantes de una vez, sin paginar (comprobado con varios eventos).
    Se usa solo para contar finishers por sexo -- nunca se persiste fila a
    fila (ver celda de arriba)."""
    data = obtener_json(f"{API_CRONOMANCHA}/races/{urllib.parse.quote(slug_carrera)}/results?limit=0")
    return data.get("data", []) if data else []


def obtener_indice_legacy(slug):
    """Metadatos de la carrera en el sistema legacy: nombre, fecha,
    modalidad, distancia y el TS que identifica el .zip de resultados. Un
    mismo slug puede tener varias entradas si se ha actualizado más de una
    vez -- nos quedamos con la más reciente."""
    data = obtener_json(f"{LEGACY_CRONOMANCHA}/data/{slug}/index.json")
    return data[-1] if data else None


def obtener_resultados_legacy(slug, ts):
    """Descarga el .zip de resultados del sistema legacy (ZIP estándar con
    un único JSON dentro) y devuelve la lista de participantes. Se usa solo
    para contar finishers por sexo -- nunca se persiste fila a fila (ver
    celda de arriba)."""
    if not ts:
        return []
    url = f"{LEGACY_CRONOMANCHA}/data/{slug}/{ts}_result.json.zip"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  \U0001f6a8 Error al descargar el zip legacy de {slug}: {e}")
        return []
    try:
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            nombre_interno = zf.namelist()[0]
            with zf.open(nombre_interno) as f:
                return json.load(f)
    except (zipfile.BadZipFile, json.JSONDecodeError) as e:
        print(f"  \U0001f6a8 Zip legacy ilegible de {slug}: {e}")
        return []


def _es_finisher_legacy(participante):
    """Un participante del sistema legacy se cuenta como finisher si no ha
    abandonado, no ha sido descalificado y tiene un tiempo de carrera
    registrado (los DNS -- inscritos que no llegaron a salir -- no tienen
    ninguno de los dos campos)."""
    return (
        participante.get("AbandonaT05") == "False"
        and participante.get("DescalificadoT05") == "False"
        and bool(participante.get("MiliSegCarreraT05"))
    )


def _fecha_legacy_a_iso(texto_fecha):
    """Convierte 'DD/MM/AAAA HH:MM:SS' (formato del sistema legacy) a
    'AAAA-MM-DD'. Devuelve None si no se puede parsear."""
    if not texto_fecha:
        return None
    try:
        return datetime.strptime(texto_fecha.split(" ")[0], "%d/%m/%Y").strftime("%Y-%m-%d")
    except ValueError:
        return None


def filas_de_evento_legacy(id_evento, slug):
    """Devuelve una fila AGREGADA (finisher_h, finisher_d, total_finishers,
    lat, lon) para un evento del sistema legacy de Cronomancha. A diferencia
    del sistema nuevo, aquí cada distancia/categoría es un `id_timingsys`
    distinto -- un slug es siempre una sola carrera, nunca varias."""
    info = obtener_indice_legacy(slug)
    if info is None:
        return []
    lat, lon = obtener_coordenadas_evento(id_evento)
    participantes = obtener_resultados_legacy(slug, info.get("TS"))

    finisher_h = sum(
        1 for p in participantes
        if _es_finisher_legacy(p) and p.get("SexoT05") == "M"
    )
    finisher_d = sum(
        1 for p in participantes
        if _es_finisher_legacy(p) and p.get("SexoT05") == "F"
    )

    nombre_carrera = info.get("NombreCarreraT01")
    distancia = info.get("DistanciaMetrosTotalT01")
    return [{
        "id_timingsys": id_evento,
        "hash_cronomancha": slug,
        "nombre_evento": nombre_carrera,
        "fecha_evento": _fecha_legacy_a_iso(info.get("FechaCarreraT01")),
        "nombre_carrera": nombre_carrera,
        "slug_carrera": slug,
        "modalidad": info.get("Modalidad"),
        "distancia_metros": int(distancia) if distancia else None,
        "lat": lat,
        "lon": lon,
        "finisher_h": finisher_h,
        "finisher_d": finisher_d,
        "total_finishers": finisher_h + finisher_d,
    }]


def filas_de_evento(id_evento, hash_evento):
    """Devuelve una fila AGREGADA por carrera del evento (finisher_h,
    finisher_d, total_finishers, lat, lon), igual que el resto de scrapers
    del proyecto. No incluye ningún dato a nivel de participante individual."""
    info_evento, carreras = obtener_carreras_evento(hash_evento)
    if info_evento is None:
        return []
    lat, lon = obtener_coordenadas_evento(id_evento)
    filas = []
    for carrera in carreras:
        resultados = obtener_resultados_carrera(carrera["race_slug"])
        finisher_h = sum(
            1 for r in resultados
            if r.get("status") == "FINISHED" and r.get("gender") == "M"
        )
        finisher_d = sum(
            1 for r in resultados
            if r.get("status") == "FINISHED" and r.get("gender") == "F"
        )
        filas.append({
            "id_timingsys": id_evento,
            "hash_cronomancha": hash_evento,
            "nombre_evento": info_evento["nombre_evento"],
            "fecha_evento": info_evento["fecha_evento"],
            "nombre_carrera": carrera["display_name"],
            "slug_carrera": carrera["race_slug"],
            "modalidad": carrera.get("modality"),
            "distancia_metros": carrera.get("total_distance_meters"),
            "lat": lat,
            "lon": lon,
            "finisher_h": finisher_h,
            "finisher_d": finisher_d,
            "total_finishers": finisher_h + finisher_d,
        })
        time.sleep(PAUSA_API)
    return filas


In [12]:
def descargar_resultados_pendientes(df_catalogo, max_eventos=None):
    """Para cada evento del catálogo con resultados publicados y no
    procesado aún (según el checkpoint), resuelve en qué sistema de
    Cronomancha están (nuevo o legacy) y descarga sus resultados agregados,
    añadiéndolos a DF_TIMINGSYS_SUCIO.csv evento a evento. `max_eventos`
    limita cuántos eventos NUEVOS se procesan en esta llamada (para la
    prueba rápida); None = todos los pendientes.

    Para el sistema legacy, `resolver_enlace_cronomancha` puede devolver
    varias carreras "hermanas" (distintas distancias del mismo evento que
    Cronomancha guarda por separado) -- se procesan todas y sus filas se
    van acumulando para ese mismo id_timingsys."""
    procesados = set(json.loads(CHECKPOINT_EVENTOS.read_text())) if CHECKPOINT_EVENTOS.exists() else set()

    pendientes = df_catalogo[
        df_catalogo["tiene_resultados"] & ~df_catalogo["id_timingsys"].isin(procesados)
    ][["id_timingsys", "nombre_evento", "fecha_texto"]].to_dict("records")
    if max_eventos is not None:
        pendientes = pendientes[:max_eventos]
    print(f"Eventos con resultados a procesar en esta llamada: {len(pendientes)}")

    for evento in pendientes:
        id_evento = evento["id_timingsys"]
        fecha_iso = fecha_catalogo_a_iso(evento.get("fecha_texto"))
        sistema, referencia = resolver_enlace_cronomancha(
            id_evento, evento.get("nombre_evento"), fecha_iso
        )
        if referencia is None:
            print(f"  \u26a0\ufe0f [{id_evento}] sin enlace a Cronomancha (nuevo o legacy), se marca como procesado y se salta")
            procesados.add(id_evento)
            CHECKPOINT_EVENTOS.write_text(json.dumps(sorted(procesados)))
            continue

        if sistema == "nuevo":
            filas = filas_de_evento(id_evento, referencia)
        else:
            filas = []
            for slug in referencia:
                filas.extend(filas_de_evento_legacy(id_evento, slug))
        print(f"  \u21b3 [{id_evento}] {sistema}/{referencia}: {len(filas)} filas")

        if filas:
            df_nuevo = pd.DataFrame(filas)
            if CSV_RESULTADOS.exists():
                df_nuevo.to_csv(CSV_RESULTADOS, mode="a", header=False, index=False, encoding="utf-8-sig")
            else:
                df_nuevo.to_csv(CSV_RESULTADOS, index=False, encoding="utf-8-sig")

        procesados.add(id_evento)
        CHECKPOINT_EVENTOS.write_text(json.dumps(sorted(procesados)))
        time.sleep(PAUSA_TIMINGSYS)

    return pd.read_csv(CSV_RESULTADOS) if CSV_RESULTADOS.exists() else pd.DataFrame()

### Ejecución Nivel 2

Igual que en el Nivel 1: con `MODO_PRUEBA = True` solo procesa `MAX_EVENTOS_RESULTADOS` eventos nuevos (rápido, para comprobar esta noche que la resolución del hash y la API siguen funcionando); con `False`, todos los pendientes según el catálogo descargado arriba. El checkpoint (`timingsys_eventos_checkpoint.json`) hace que los eventos ya procesados en la prueba NO se repitan al pasar a la descarga completa -- simplemente se suma el resto.

Cada carrera añade una única fila agregada a `DF_TIMINGSYS_SUCIO.csv` (`finisher_h`, `finisher_d`, `total_finishers`), nunca una fila por participante.

In [15]:
df_timingsys = descargar_resultados_pendientes(df_catalogo, max_eventos=MAX_EVENTOS_RESULTADOS)
print(f"DF_TIMINGSYS_SUCIO.csv: {df_timingsys.shape}")
print(df_timingsys[["id_timingsys", "nombre_carrera", "lat", "lon", "finisher_h", "finisher_d", "total_finishers"]].to_string())


Eventos con resultados a procesar en esta llamada: 0
DF_TIMINGSYS_SUCIO.csv: (394, 13)
     id_timingsys                                                     nombre_carrera        lat       lon  finisher_h  finisher_d  total_finishers
0             767                    XXI CP SANTA EULALIA - VILLARES DEL SAZ - 2026         NaN       NaN         136          47              183
1             766                                V CP LA JOYA BARROCA - SISANTE 2026        NaN       NaN         162          56              218
2             765                      XXII CP SAN GIL - MOTILLA DEL PALANCAR - 2026        NaN       NaN         208          95              303
3             797                           IX LEGUA ALMARCHEÑA - LA ALMARCHA - 2026  39.686041 -2.379204          56          14               70
4             757               XIV MARCHA MTB V. FUENTELESPINO DE HARO 2026 - CORTA  39.689785 -2.669688           5           0                5
5             757              

### Resumen para esta noche

`MODO_PRUEBA` está ya en `False`: al hacer Run All esta noche se lanza directamente la descarga completa (las páginas de catálogo que falten y todos los eventos con resultados pendientes -- los 757 y 797 ya procesados no se repiten, gracias al checkpoint). Si quieres una prueba rápida primero, pon `MODO_PRUEBA = True` en la celda del interruptor.

`DF_TIMINGSYS_SUCIO.csv` queda con una fila por carrera (no por participante): `id_timingsys, hash_cronomancha, nombre_evento, fecha_evento, nombre_carrera, slug_carrera, modalidad, distancia_metros, lat, lon, finisher_h, finisher_d, total_finishers` -- el mismo tipo de esquema agregado que `Scraper_Kronoak` y `scrape_cronorunner` (sin nombres, dorsales ni ningún dato de participante individual), más `lat`/`lon` del evento sacadas de su página de ubicación, igual que hace `scrape_raceresult`. Quedan listas para geocodificar a `municipio`/`comarca`/`provincia` en la limpieza, cuando `lat`/`lon` no sean nulas (el organizador no siempre marca el punto en el mapa).

Si se interrumpe a medias (Kernel → Interrupt, se cierra el portátil, un error de red...), basta con volver a ejecutar las mismas dos celdas: `descargar_catalogo_completo()` retoma por página y `descargar_resultados_pendientes()` retoma por evento -- no repiten lo ya hecho.

**Añadido después de la descarga de esta noche:** se ha incorporado soporte para el sistema legacy de Cronomancha (`resultados.cronomancha.com`), que resultó ser donde estaban la mayoría de los eventos que antes salían como "sin enlace a Cronomancha". Para recuperarlos hay que volver a ejecutar `descargar_resultados_pendientes()` tras quitar del checkpoint (`timingsys_eventos_checkpoint.json`) los IDs que se saltaron solo por falta de soporte legacy -- los que de verdad no tienen resultados en ningún sistema de Cronomancha (circuitos propios, páginas en vivo, u otro proveedor de cronometraje) se quedan marcados como procesados.


### Reintento de los eventos que se marcaron como procesados sin dejar ninguna fila

En la ejecución completa, 468 eventos del catálogo tenían resultados, pero solo 379 acabaron con filas en `DF_TIMINGSYS_SUCIO.csv`. Los otros 89 se marcaron como procesados igualmente (sin enlace a Cronomancha en ese momento, o con el sistema legacy todavía sin soportar) y por eso `descargar_resultados_pendientes()` ya no los vuelve a intentar.

Esta celda quita del checkpoint **solo esos eventos** (los que están en el checkpoint, tienen resultados en el catálogo y no aparecen en el CSV) y los vuelve a procesar. Los 379 que ya tienen filas no se tocan, así que no se duplica nada. Los que sigan sin resolverse (circuitos propios, páginas en vivo, otro proveedor) se volverán a marcar como procesados y saltados, sin coste.

In [ ]:
df_actual = pd.read_csv(CSV_RESULTADOS)
df_catalogo = pd.read_csv(CSV_CATALOGO)

con_filas = set(df_actual["id_timingsys"])
sin_filas = set(df_catalogo.loc[df_catalogo["tiene_resultados"], "id_timingsys"]) - con_filas

procesados = set(json.loads(CHECKPOINT_EVENTOS.read_text()))
CHECKPOINT_EVENTOS.write_text(json.dumps(sorted(procesados - sin_filas)))
print(f"{len(sin_filas)} eventos con resultados y sin filas quitados del checkpoint")

df_timingsys = descargar_resultados_pendientes(df_catalogo, max_eventos=MAX_EVENTOS_RESULTADOS)
print(f"DF_TIMINGSYS_SUCIO.csv: {df_actual.shape} -> {df_timingsys.shape}")